In [ ]:
import openai
import os

with open('.openai_key', 'r') as f:
    os.environ['OPENAI_API_KEY'] = f.read()

client = openai.OpenAI()

In [ ]:
REQUEST_BATCH_PATH = 'request_batches'
RESPONSE_BATCH_PATH = 'response_batches'

os.makedirs(REQUEST_BATCH_PATH, exist_ok=True)
os.makedirs(RESPONSE_BATCH_PATH, exist_ok=True)

### Upload Batch Input Files

In [ ]:
categories = os.listdir(REQUEST_BATCH_PATH)

In [ ]:
batch_input_files = []
for category in categories:
    category_path = os.path.join(REQUEST_BATCH_PATH, category)
    batch_input_files.append(client.files.create(
        file=open(category_path, 'rb'),
        purpose='batch'
    ))

### Create Batches

In [ ]:
batches = []

for batch_input_file in batch_input_files:
    batches.append(client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint='/v1/responses',
        completion_window='24h',
        metadata={
            'category': batch_input_file.filename
        }
    ))

### Monitor batches

In [ ]:
batches_list = client.batches.list(limit=50)

for cur_batch in batches_list.data:
    category = cur_batch.metadata['category']
    status = cur_batch.status
    total = cur_batch.request_counts.total
    completed = cur_batch.request_counts.completed
    out_file = cur_batch.output_file_id
    print(f'{category.ljust(30)}: \t{status} ({completed}/{total}) -> {out_file}')

### Download and Save Results

In [ ]:
batches_list = client.batches.list(limit=50)

for cur_batch in batches_list.data:
    category = cur_batch.metadata['category']
    out_file = cur_batch.output_file_id

    print(f'Saving {category}...')
    file_response = client.files.content(out_file)

    with open(os.path.join(RESPONSE_BATCH_PATH, category), 'w') as f:
        f.write(file_response.text)